In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
# set the base path 
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Epithelial_annotation_noHarmony.h5ad'))

In [ ]:
# Look at iCMS2 and iCMS3 scores 
df = pd.read_csv('/home/cporter/atlas_remake_Aug_26_2024/docs/41588_2022_1100_MOESM3_ESM.csv', keep_default_na=False) # UPDATE THIS 
subsets = df.columns
print(subsets)
for s in subsets:
    sc.tl.score_genes(adata, df[s].values[0:df.shape[0]], score_name = s, use_raw=True)

sc.set_figure_params(figsize=(4, 4))
fig, ax = plt.subplots()
sc.pl.umap(adata, color='iCMS2_Up', size=1, ax=ax, show=True, cmap='inferno')
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/iCMS2_Up_noHarmony_epi.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

sc.set_figure_params(figsize=(4, 4))
fig, ax = plt.subplots()
sc.pl.umap(adata, color='iCMS3_Up', size=1, ax=ax, show=True, cmap='inferno')
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/iCMS3_Up_noHarmony_epi.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# create dataframe with mean iCMS2/3 up scores for statistical testing 
rows = []
labels = []

iCMS = adata.obs[['iCMS2_Up','iCMS3_Up']]

for i in np.unique(adata.obs['FRID']):
    index = adata.obs['FRID'] == i
    cohort = np.unique(adata.obs['Cohort'][index])[0]
    decade = np.unique(adata.obs['Decade'][index])[0]
    age = np.unique(adata.obs['Age'][index])[0]
    stage = np.unique(adata.obs['Overall_Stage'][index])[0]
    sex = np.unique(adata.obs['Sex'][index])[0]
    treatment = np.unique(adata.obs['Therapy_v2'][index])[0]
    side = np.unique(adata.obs['Sidedness'][index])[0]
    msi = np.unique(adata.obs['MSI_v2'][index])[0]


    mean_rows = iCMS[index].mean(axis=0)
    row_with_meta = list(mean_rows) + [cohort, decade, age, stage, sex, treatment, side, msi]
    rows.append(row_with_meta)
    labels.append(i)

column_names = list(iCMS.columns) + ['Cohort', 'Decade', 'Age', 'Stage', 'Sex', 'Treatment', 'Side', 'MSI']
df = pd.DataFrame(rows, index=labels, columns=column_names)

In [ ]:
df

In [ ]:
import statsmodels.formula.api as smf

# remove unknown columns 
df_noUNK = df[df['Stage'] != 'Not Applicable']
df_noUNK = df_noUNK[df_noUNK['MSI'] != 'Unknown']

model = smf.ols(
    formula="iCMS2_Up ~ age_scale + C(MSI) + C(Sex) + C(Stage) + C(Side) + C(Treatment)", 
    data=df_noUNK
).fit()

print(model.summary())

In [ ]:
# remove unknown columns 
df_noUNK = df[df['Stage'] != 'Not Applicable']
df_noUNK = df_noUNK[df_noUNK['MSI'] != 'Unknown']

model = smf.ols(
    formula="iCMS3_Up ~ age_scale + C(MSI) + C(Sex) + C(Stage) + C(Side) + C(Treatment)", 
    data=df_noUNK
).fit()

print(model.summary())